# Fetch and Embed recent posts for pertinent users

## Imports and clients and reusables

In [2]:
from utils.schemas import Content
# Import additional dependencies for AI analysis
from openai import OpenAI
import instructor
import os
api_key=os.getenv("OPENAI_API_KEY")

# Initialize OpenAI client with instructor for structured outputs
openai_client = instructor.from_openai(OpenAI(api_key=api_key))

from atproto import Client
import os
from dotenv import load_dotenv

# Load environment variables (create a .env file with your credentials)
load_dotenv()

# Initialize the Bluesky client
bluesky_client = Client()

# Authenticate using your handle and app password
BLUESKY_HANDLE = os.getenv('BLUESKY_HANDLE')  # e.g., 'username.bsky.social'
BLUESKY_APP_PASSWORD = os.getenv('BLUESKY_APP_PASSWORD')  # Your app password

# Login to Bluesky
bluesky_client.login(BLUESKY_HANDLE, BLUESKY_APP_PASSWORD)
print(f"✅ Successfully logged in as {BLUESKY_HANDLE}")

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


✅ Successfully logged in as thedevguild.bsky.social


In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import os
from dotenv import load_dotenv
import openai
load_dotenv()
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

In [5]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

In [6]:
CONTENT_COLLECTION_NAME="Content-collection-00"

In [26]:
def process_and_upsert_twitter_content(input: list[Content], batch_size: int = 100):
    """
    Upserts tweet Content objects to Qdrant in batches to avoid "payload too large" errors.

    Args:
        input (list[Content]): List of Content objects representing tweets.
        batch_size (int): Number of points to upsert in each batch.
    """
    # get latest collection size
    collection_size = qdrant_client.count(collection_name=CONTENT_COLLECTION_NAME)
    current_id = collection_size.count + 1

    total_points = 0
    for batch_start in range(0, len(input), batch_size):
        batch = input[batch_start : batch_start + batch_size]
        valid_items = [
            data for data in batch
            if isinstance(data.content_text, str) and data.content_text.strip()
        ]
        if not valid_items:
            print(f"Batch {batch_start // batch_size + 1} skipped (no valid content_text)")
            continue

        text_to_embed = [data.content_text for data in valid_items]
        embeddings = get_embeddings_batch(text_to_embed)

        pointstructs = []

        for embedding, data in zip(embeddings, batch):
            pointstructs.append(
                PointStruct(
                    id=current_id,
                    vector={
                        "text-embedding-3-small": embedding,
                        "bm25": Document(
                            text=data.content_text,
                            model="qdrant/bm25"
                        ),
                    },
                    payload=data.model_dump(),
                )
            )
            current_id += 1

        # Upsert this batch
        qdrant_client.upsert(
            collection_name=CONTENT_COLLECTION_NAME,
            points=pointstructs
        )
        total_points += len(pointstructs)
        print(f"Upserted batch {batch_start // batch_size + 1}: {len(pointstructs)} points")

    print(f"Upserted total {total_points} points for {len(input)} rows")

## Fetch recent posts

In [13]:
def get_recent_posts(client, user_handle, limit=50) -> list[Content]:
    """
    Get recent posts for a user, extracting the correct created_date from the post record.
    """
    from dateutil import parser as date_parser

    # Fetch user DID if necessary
    profile = client.app.bsky.actor.get_profile({'actor': user_handle})
    user_did = profile['did']

    # Fetch the user's feed/posts using the feed API
    response = client.app.bsky.feed.get_author_feed({'actor': user_did, 'limit': limit})

    feed_items = getattr(response, 'feed', [])

    posts = []
    for item in feed_items:
        post = getattr(item, 'post', None)
        if post is None:
            continue

        # The post, record, and author may be dataclass/attrs or dicts, handle both
        # 1. Get post_data as dict for fallback, but prefer attribute access
        post_data = post.__dict__ if hasattr(post, '__dict__') else post
        # 2. Get record
        record = getattr(post, 'record', None) or post_data.get('record', {})
        record_data = record.__dict__ if hasattr(record, '__dict__') else record
        # 3. Get author
        author = getattr(post, 'author', None) or post_data.get('author', {})
        author_data = author.__dict__ if hasattr(author, '__dict__') else author

        # Extract created_at: it is always 'created_at' (snake_case) in the atproto library
        created_at_str = None
        if isinstance(record_data, dict):
            created_at_str = record_data.get('created_at')
        else:
            created_at_str = getattr(record_data, 'created_at', None)

        created_at_val = None
        if created_at_str:
            try:
                created_at_val = date_parser.parse(created_at_str)
            except Exception:
                created_at_val = created_at_str

        # Defensive fallback for text, id, likes, etc
        if isinstance(record_data, dict):
            content_text = record_data.get('text', '')
        else:
            content_text = getattr(record_data, 'text', '')

        if isinstance(post_data, dict):
            post_id = post_data.get('uri', '')
            likes = post_data.get('like_count', 0)
        else:
            post_id = getattr(post, 'uri', '')
            likes = getattr(post, 'like_count', 0)

        if isinstance(author_data, dict):
            author_handle = author_data.get('handle', user_handle)
        else:
            author_handle = getattr(author_data, 'handle', user_handle)

        posts.append(
            Content(
                id=post_id,
                content_text=content_text,
                mediaType='bluesky',
                author=author_handle,
                created_date=created_at_val,
                likes=likes if likes is not None else 0
            )
        )
    return posts

In [14]:
test_handle = "thedevguild.bsky.social"
posts = get_recent_posts(bluesky_client, test_handle)

print(f"Number of posts: {len(posts)}\n")

print("First 5 posts:")
for i, post in enumerate(posts[:5], start=1):
    print(f"\nPost {i}:")
    print(f"  id: {post.id}")
    print(f"  author: {post.author}")
    print(f"  created_date: {post.created_date}")
    print(f"  likes: {post.likes}")
    print(f"  content_text: {post.content_text}")
    print(f"  mediaType: {post.mediaType}\n")

Number of posts: 14

First 5 posts:

Post 1:
  id: at://did:plc:a5nmb42bv7wuvjbkdlw2q3bs/app.bsky.feed.post/3m7pdzmjaxs24
  author: thedevguild.bsky.social
  created_date: 2025-12-11 10:20:40.323000+00:00
  likes: 1
  content_text: In the age of AI, trust in online information is crucial yet challenging. That’s why on-chain reputation systems are essential! With community-verified badges and attestations, we can confidently recognize and showcase each other's skills. Let’s build trust together! 🌐💪 #TheGuild #Web3
  mediaType: bluesky


Post 2:
  id: at://did:plc:a5nmb42bv7wuvjbkdlw2q3bs/app.bsky.feed.post/3m7nnin4rss24
  author: thedevguild.bsky.social
  created_date: 2025-12-10 18:04:48.479000+00:00
  likes: 0
  content_text: 🌍 The Guild knows no boundaries! We empower developers across the globe to collaborate, learn, and create together. Join us and be part of a community that celebrates diversity and innovation in tech! 🚀✨ #TheGuild #GlobalDev #Collaboration
  mediaType: bluesky




## Embed them

In [15]:
process_and_upsert_twitter_content(posts)

Upserted batch 1: 14 points
Upserted total 14 points for 14 rows


## Full flow get posts, embed and upsert

In [24]:
def process_profile_posts_into_qdrant(client, handle: str, limit: int = 50):
    posts = get_recent_posts(client, handle, limit)

    print(f"Number of posts: {len(posts)}\n")

    print("First 5 posts:")
    for i, post in enumerate(posts[:5], start=1):
        print(f"\nPost {i}:")
        print(f"  id: {post.id}")
        print(f"  author: {post.author}")
        print(f"  created_date: {post.created_date}")
        print(f"  likes: {post.likes}")
        print(f"  content_text: {post.content_text}")
        print(f"  mediaType: {post.mediaType}\n")
    process_and_upsert_twitter_content(posts)

In [28]:
process_profile_posts_into_qdrant(bluesky_client,"vortexofadigitalkind.com")

Number of posts: 50

First 5 posts:

Post 1:
  id: at://did:plc:ckrvar23cqkfyaiatleirngu/app.bsky.feed.post/3m7krjakqx225
  author: vortexofadigitalkind.com
  created_date: 2025-12-09 14:38:44.613000+00:00
  likes: 0
  content_text: My #Web3 #blog site is getting close to becoming a live thing. Hoping to have the new site up before the new year to replace #Wordpress.  vortexofadigitalkind.com
  mediaType: bluesky


Post 2:
  id: at://did:plc:ckrvar23cqkfyaiatleirngu/app.bsky.feed.post/3m7dkp2emuc27
  author: vortexofadigitalkind.com
  created_date: 2025-12-06 17:48:05.084000+00:00
  likes: 1
  content_text: Thank you @missourihashish.bsky.social @madeindex.bsky.social @opengraph.tools for the recent follows.
  mediaType: bluesky


Post 3:
  id: at://did:plc:ckrvar23cqkfyaiatleirngu/app.bsky.feed.post/3m7daszwqzk27
  author: vortexofadigitalkind.com
  created_date: 2025-12-06 14:51:21.430000+00:00
  likes: 0
  content_text: I've just retired the sounds notifications from my phone. Such 